In [2]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("Qwen/Qwen3-0.6B")

print("Model loaded!")
print(f"Layers: {model.cfg.n_layers}")
print(f"Hidden dimension: {model.cfg.d_model}")

/tmp/ipykernel_12318/1457788674.py:3: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("Qwen/Qwen3-0.6B")
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen3-0.6B into HookedTransformer
Model loaded!
Layers: 28
Hidden dimension: 1024


In [10]:
prompt = """Solve this problem step by step.

Alice is older than Bob.
Bob is older than Charlie.

Question: Who is the oldest?

Explain your reasoning and then give the final answer."""

tokens = model.to_tokens(prompt)

logits, cache = model.run_with_cache(tokens)

print("Number of tokens:", tokens.shape[1])
print("Cached activation keys:", len(cache))

Number of tokens: 38
Cached activation keys: 677


In [11]:
for layer in [0, 7, 14, 21, 27]:
    activation = cache[f"blocks.{layer}.hook_resid_post"]
    print(
        f"Layer {layer}:",
        activation.shape
    )

Layer 0: torch.Size([1, 38, 1024])
Layer 7: torch.Size([1, 38, 1024])
Layer 14: torch.Size([1, 38, 1024])
Layer 21: torch.Size([1, 38, 1024])
Layer 27: torch.Size([1, 38, 1024])


In [4]:
clean_prompt = """Alice is older than Bob.
Bob is older than Charlie.
Who is the oldest?"""

corrupt_prompt = """Alice is younger than Bob.
Bob is older than Charlie.
Who is the oldest?"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

print("Clean:")
print(model.to_str_tokens(clean_tokens))

print("\nCorrupted:")
print(model.to_str_tokens(corrupt_tokens))

Clean:
['Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n', 'Who', ' is', ' the', ' oldest', '?']

Corrupted:
['Alice', ' is', ' younger', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n', 'Who', ' is', ' the', ' oldest', '?']


In [5]:
clean_logits = model(clean_tokens)
corrupt_logits = model(corrupt_tokens)

clean_next = clean_logits[0, -1].argmax()
corrupt_next = corrupt_logits[0, -1].argmax()

print("Clean prediction:", repr(model.to_string(clean_next)))
print("Corrupted prediction:", repr(model.to_string(corrupt_next)))

Clean prediction: ' A'
Corrupted prediction: ' A'


In [6]:
clean_prompt = """Answer the question with exactly one name.

Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer: Alice"""

corrupt_prompt = """Answer the question with exactly one name.

Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer: Bob"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

print("Clean:")
print(model.to_str_tokens(clean_tokens))

print("\nCorrupted:")
print(model.to_str_tokens(corrupt_tokens))

Clean:
['Answer', ' the', ' question', ' with', ' exactly', ' one', ' name', '.\n\n', 'Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Answer', ':', ' Alice']

Corrupted:
['Answer', ' the', ' question', ' with', ' exactly', ' one', ' name', '.\n\n', 'Alice', ' is', ' younger', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Answer', ':', ' Bob']


In [10]:
clean_prompt = """Answer with exactly one name.

Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer:"""

corrupt_prompt = """Answer with exactly one name.

Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer:"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

clean_logits = model(clean_tokens)
corrupt_logits = model(corrupt_tokens)

# Probability of the two candidate answers
for name in [" Alice", " Bob"]:
    token = model.to_single_token(name)
    clean_prob = clean_logits[0, -1].softmax(-1)[token].item()
    corrupt_prob = corrupt_logits[0, -1].softmax(-1)[token].item()

    print(f"{name}: clean={clean_prob:.4f}, corrupted={corrupt_prob:.4f}")

 Alice: clean=0.1046, corrupted=0.0604
 Bob: clean=0.0052, corrupted=0.0048


## Outcome

The initial toy age-ordering task was not suitable for activation patching with Qwen3-0.6B: the model did not realiably distinguish the clean and corrupted conditions. We therefore do not interpret any activation patching results from this task.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
chat_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

messages = [
    {
        "role": "user",
        "content": """Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Explain your reasoning and give the final answer."""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

inputs = tokenizer(text, return_tensors="pt").to(chat_model.device)

outputs = chat_model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.6,
    top_p=0.95,
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print(response)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

<think>
Okay, let's try to figure out who is the oldest here. So, the problem says Alice is older than Bob, and Bob is older than Charlie. The question is asking who is the oldest. 

First, I need to remember the order of these relationships. Alice is older than Bob, and Bob is older than Charlie. So starting from the top, Alice comes first. Then Bob, and then Charlie. So if I list them in order, it would be Alice > Bob > Charlie. Therefore, Alice is the oldest among them.

Wait, but maybe I should check if there's any possibility that someone else could be older than Alice? But according to the given statements, there's no information suggesting that. Alice is directly older than Bob, and Bob is older than Charlie. So there's no contradiction here. The relationships are linear: Alice > Bob > Charlie. 

Another way to think about it: if we have a chain of inequalities, Alice is the first one, followed by Bob, then Charlie. So the oldest is Alice. 

Is there any trick here? Like, maybe 

## Qwen3 Chat-Template Sanity Check

The initial raw-text interface produced unreliable behavior on the toy
reasoning task. We therefore tested Qwen3-0.6B using its official chat
template with thinking mode enabled.

The chat-formatted model produced a coherent chain-of-thought trajectory
for the age-ordering task, suggesting that the earlier failure was at least
partly due to the inference interface rather than the model's inability to
solve the task.

Next: determine whether the same chat-formatted model can be connected to
our activation-inspection pipeline.